# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² mental health survey dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll list the `@id` for each record set, along with their available fields and columns.

In [ ]:
# Explore record sets and their fields/columns
record_sets = [rs['@id'] for rs in metadata.recordSet]
print('Available Record Sets:')
for rs_obj in metadata.recordSet:
    print(f"RecordSet @id: {rs_obj['@id']}")
    if 'field' in rs_obj:
        print('  Fields:')
        for f in rs_obj['field']:
            print(f"    Field @id: {f['@id']} - label: {f.get('label', f['@id'])}")
    if 'column' in rs_obj:
        print('  Columns:')
        for c in rs_obj['column']:
            print(f"    Column @id: {c['@id']} - label: {c.get('label', c['@id'])}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

We'll extract the main survey record set which includes psychological, demographic, and several assessment scores (like GAD-7 and PHQ-9).

In [ ]:
# Assume the primary record set is available (replace with correct @id if needed):
main_record_set_id = ''
for rs in metadata.recordSet:
    if 'Survey' in rs.get('label', '') or 'mental_health' in rs.get('name', ''):
        main_record_set_id = rs['@id']
        break
if not main_record_set_id and record_sets:
    # fallback if no explicit label, just take the first
    main_record_set_id = record_sets[0]

print(f"Using RecordSet: {main_record_set_id}")
# Load available records from the survey record set
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
print(f"Columns (@id): {df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. We'll demonstrate these with fields (referenced by their `@id`) like GAD-7 (anxiety), PHQ-9 (depression), and demographic attributes.

In [ ]:
# Choose a numeric field for filtering, such as GAD-7 score
gad7_field_id = ''
for col in df.columns:
    if 'GAD' in col or 'gad7' in col or 'anxiety_score' in col:
        gad7_field_id = col
        break
# Fallback: use first numeric column
if not gad7_field_id:
    numeric_cols = df.select_dtypes(include=np.number).columns
    gad7_field_id = numeric_cols[0] if len(numeric_cols) > 0 else df.columns[0]

print(f"Using numeric field: {gad7_field_id}")

# Filter records where GAD-7 score > threshold
threshold = 10
filtered_df = df[df[gad7_field_id] > threshold]
print(f"Filtered records with {gad7_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize GAD-7 scores
filtered_df[f"{gad7_field_id}_normalized"] = (filtered_df[gad7_field_id] - filtered_df[gad7_field_id].mean()) / filtered_df[gad7_field_id].std()
print(f"Normalized {gad7_field_id} for filtered records:")
print(filtered_df[[gad7_field_id, f"{gad7_field_id}_normalized"]].head())

# Group by a demographic field (e.g., gender, use @id)
group_field = ''
for col in df.columns:
    if 'gender' in col or 'sex' in col:
        group_field = col
        break
if not group_field:
    group_field = df.columns[0]  # fallback

if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[gad7_field_id].mean().reset_index()
    print(f"Grouped mean of {gad7_field_id} by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions and relationships using the selected fields.

In [ ]:
# Histogram of the GAD-7 Score
plt.figure(figsize=(8, 5))
sns.histplot(df[gad7_field_id].dropna(), bins=15, kde=True)
plt.title(f"Distribution of {gad7_field_id}")
plt.xlabel(gad7_field_id)
plt.ylabel('Count')
plt.show()

# Boxplot by gender (if applicable)
if group_field in df.columns and gad7_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=df[group_field], y=df[gad7_field_id])
    plt.title(f"{gad7_field_id} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(gad7_field_id)
    plt.show()

# Scatter plot of two assessment scores if available
phq9_field_id = ''
for col in df.columns:
    if 'PHQ' in col or 'phq9' in col or 'depression_score' in col:
        phq9_field_id = col
        break
if phq9_field_id and gad7_field_id in df.columns:
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=df[gad7_field_id], y=df[phq9_field_id])
    plt.title(f"{gad7_field_id} vs {phq9_field_id}")
    plt.xlabel(gad7_field_id)
    plt.ylabel(phq9_field_id)
    plt.show()

## 6. Conclusion
In this notebook, we've loaded the FAIR² mental health survey dataset from Kilifi County, explored its structure using `mlcroissant`, and performed basic data analysis including filtering, normalization, grouping, and visualizations. This dataset is valuable for understanding mental health indicators and demographic differences in Kilifi, Kenya and is AI-ready for further machine learning applications.

Key findings:
- The dataset records contain detailed survey responses with assessment scores for GAD-7, PHQ-9, and demographic information referenced by `@id`.
- Exploratory analysis reveals potential differences in mental health scores by demographic group.
- Data is well-structured for reproducible research and public health strategy analysis.